<a href="https://colab.research.google.com/github/Rouba-Os/AI-Course/blob/main/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Load Dataset
import pandas as pd

df = pd.read_csv('final_cleaned_data.csv', parse_dates=['date'], index_col='date')

print("=== Dataset Preview ===")
print(df.head())

In [ ]:
# Sort Data
df = df.sort_values(by=['ticker','date'])

In [ ]:
# Feature Engineering
## MACD Calculation
df['EMA12'] = df.groupby('ticker')['close'].transform(
    lambda x: x.ewm(span=12, adjust=False).mean())

df['EMA26'] = df.groupby('ticker')['close'].transform(
    lambda x: x.ewm(span=26, adjust=False).mean())

df['MACD'] = df['EMA12'] - df['EMA26']

df['Signal_Line'] = df.groupby('ticker')['MACD'].transform(
    lambda x: x.ewm(span=9, adjust=False).mean())

print("MACD Calculated")
print(df[['MACD','Signal_Line']].head())


In [ ]:
## MACD Signals
df['MACD_signal'] = 0

df.loc[df['MACD'] > df['Signal_Line'], 'MACD_signal'] = 1   # Buy
df.loc[df['MACD'] < df['Signal_Line'], 'MACD_signal'] = -1  # Sell


In [ ]:
## # RSI Calculation
delta = df.groupby('ticker')['close'].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.groupby(df['ticker']).transform(
    lambda x: x.ewm(span=14, adjust=False).mean())

avg_loss = loss.groupby(df['ticker']).transform(
    lambda x: x.ewm(span=14, adjust=False).mean())

rs = avg_gain / avg_loss

df['RSI'] = 100 - (100 / (1 + rs))

print("RSI Calculated")
print(df['RSI'].head())

In [ ]:
## RSI Signals
df['RSI_signal'] = 0

df.loc[df['RSI'] < 30, 'RSI_signal'] = 1   # Buy
df.loc[df['RSI'] > 70, 'RSI_signal'] = -1  # Sell

In [ ]:
## Final Trading Signal
df['Final_Signal'] = 0

df.loc[
    (df['MACD_signal'] == 1) & (df['RSI_signal'] == 1),
    'Final_Signal'
] = 1

df.loc[
    (df['MACD_signal'] == -1) & (df['RSI_signal'] == -1),
    'Final_Signal'
] = -1

In [ ]:
# Data Preparation
df.dropna(inplace=True)

X = df[['MACD', 'Signal_Line', 'RSI']]
y = df['Final_Signal']


In [ ]:
# Train-Test Split (No Shuffle)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False)

In [ ]:
# Model Building
## Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

In [ ]:
# Model Building
## Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

In [ ]:
# Model Building
## SVM
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)

In [ ]:
# Model Evaluation
from sklearn.metrics import classification_report

for model in [lr, rf, svm]:
    preds = model.predict(X_test)
    print("\nModel:", model)
    print(classification_report(y_test, preds))

In [ ]:
# Optimization
rf = RandomForestClassifier(n_estimators=200, max_depth=10)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)

print("Optimized Random Forest")
print(classification_report(y_test, preds))

In [ ]:
# Cross Validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5)

print("Cross Validation Score")
print(scores.mean())